# AI use at work: preparing and matching the country tables

This is the publication edition of the final preparation steps from my working Colab notebook. It removes directory-inspection cells, abandoned comparisons and competing exports. It retains the original survey entries and final May Claude / Q2 OpenAI selections. Statistics are in notebook 02.

The source files are deliberately not bundled. To inspect the analysis without downloading the large files, start with notebook 02: the final country tables are included.

**Inputs:** Anthropic release 26 June 2026 (May activity); OpenAI Apr–Jun 2026; Microsoft March 2026; ILO 2025; transcribed Ipsos 2026 and Gallup pooled 2023–2025.

Run from the project folder with Python 3.12 and the packages in requirements.txt. In Colab, upload/clone the project files and configure the raw-data folder. No notebook automatically mounts or obtains access to someone else's Drive.

In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

# Run from this project folder, or set AI_CAPSTONE_PROJECT_DIR to its location.
PROJECT_ROOT = Path(os.environ.get("AI_CAPSTONE_PROJECT_DIR", str(Path.cwd()))).resolve()
assert (PROJECT_ROOT / "data_clean" / "survey_values.csv").exists(), (
    "Set PROJECT_ROOT to the project folder containing data_clean/survey_values.csv."
)
CLEAN = PROJECT_ROOT / "data_clean"
RESULTS = PROJECT_ROOT / "results"
RESULTS.mkdir(exist_ok=True)
print("Project:", PROJECT_ROOT.name)

Project: 04_AI_at_Work_Tableau_Google_Colab


## 1. Locate the raw source files

The default is data_raw/ inside this project. To use my existing Drive archive in Colab, I can mount my own Drive and point RAW_DIR to its 01_raw_data folder. Keep the original filenames. The country-only Anthropic extract is an alternative input, not a second dataset to concatenate.

In [2]:
import pycountry
RAW_DIR = Path(os.environ.get("AI_CAPSTONE_RAW_DIR", str(PROJECT_ROOT / "data_raw")))
anthropic_path = RAW_DIR / "anthropic" / "aei_claude_ai_2026-06-26.csv"
if not anthropic_path.exists():
    anthropic_path = RAW_DIR / "anthropic" / "anthropic_country_only.csv"
openai_dir = RAW_DIR / "openai"
if not openai_dir.exists():
    openai_dir = RAW_DIR / " openai"
paths = {
    "Anthropic": anthropic_path,
    "OpenAI rank": openai_dir / "share_of_messages_by_country_quarter_rank.csv",
    "OpenAI work": openai_dir / "share_of_messages_by_work_related_country_month.csv",
    "OpenAI doing": openai_dir / "share_of_messages_by_work_related_ask_do_express_country_month.csv",
    "Microsoft": RAW_DIR / "microsoft" / "estimated-share-people-generative-ai.csv",
    "ILOSTAT": RAW_DIR / "ilostat" / "GDP_2HRW_NOC_NB_A-20261008T2240.csv",
}
missing = [name for name, path in paths.items() if not path.is_file()]
assert not missing, f"Missing source files: {missing}. See data_raw/README.md."
print("All six input files found.")

All six input files found.


## 2. Standardise the country keys

ISO-3 codes are the join key. OpenAI's ISO-2 code NA means Namibia, so read it as text instead of converting it to a missing value. The country-name exceptions below come from the original cleaning work.

In [3]:
manual_fixes = {
    "Türkiye": "TUR", "Czechia": "CZE", "Republic of Korea": "KOR",
    "Viet Nam": "VNM", "Taiwan, China": "TWN",
    "United States of America": "USA",
    "United Kingdom of Great Britain and Northern Ireland": "GBR",
    "Côte d'Ivoire": "CIV", "Tanzania, United Republic of": "TZA",
    "Bolivia, Plurinational State of": "BOL",
}
def name_to_iso3(name):
    if name in manual_fixes:
        return manual_fixes[name]
    try:
        return pycountry.countries.lookup(name).alpha_3
    except LookupError:
        return None

def iso2_to_iso3(value):
    item = pycountry.countries.get(alpha_2=str(value))
    return item.alpha_3 if item else None

assert iso2_to_iso3("NA") == "NAM"
survey = pd.read_csv(CLEAN / "survey_values.csv", keep_default_na=False)
assert len(survey) == 31 and survey["iso3"].is_unique
print("Survey countries:", len(survey))

Survey countries: 31


## 3. Select Claude usage and task measures

Usage is May-only and country-level. Task measures use the overall category; other categories would duplicate country-metric rows. Estimated time reduction converts the human-only hours to minutes before comparing with the AI-assisted minutes. This is a model estimate for tasks, not measured employee hours.

In [4]:
columns = ["date_start", "geo_id", "geo_level", "category_name", "metric_id", "value"]
country = pd.read_csv(paths["Anthropic"], usecols=columns, keep_default_na=False)
country = country[country["geo_level"].eq("country")].copy()
country["value"] = pd.to_numeric(country["value"], errors="raise")
may = country[country["date_start"].astype(str).str[:10].eq("2026-05-01")]
usage_rows = may[may["metric_id"].isin(["usage_per_capita_index", "usage_pct"])]
assert not usage_rows.duplicated(["geo_id", "metric_id"]).any()
claude = usage_rows.pivot(index="geo_id", columns="metric_id", values="value").reset_index()
claude.columns.name = None
claude = claude.rename(columns={"geo_id":"iso3", "usage_pct":"claude_usage_share_pct",
                                "usage_per_capita_index":"claude_per_capita_index"})

depth_metrics = ["use_case_work_pct", "collaboration_bucket_automation_pct", "ai_autonomy_mean",
                 "human_only_time_mean", "human_with_ai_time_mean"]
d = may[may["category_name"].eq("overall") & may["metric_id"].isin(depth_metrics)]
assert not d.duplicated(["geo_id", "metric_id"]).any()
depth = d.pivot(index="geo_id", columns="metric_id", values="value").reset_index()
depth.columns.name = None
assert (depth["human_only_time_mean"] > 0).all()
depth["claude_est_time_saved_pct"] = 100 * (
    1 - depth["human_with_ai_time_mean"] / (depth["human_only_time_mean"] * 60)
)
depth = depth.rename(columns={"geo_id":"iso3", "use_case_work_pct":"claude_work_pct",
                              "collaboration_bucket_automation_pct":"claude_automation_pct",
                              "ai_autonomy_mean":"claude_autonomy"})
depth = depth[["iso3", "claude_work_pct", "claude_automation_pct", "claude_autonomy", "claude_est_time_saved_pct"]]
print("Claude countries:", len(claude))
assert len(claude) == 121

Claude countries: 121


## 4. Select the OpenAI, Microsoft and ILO periods

OpenAI use is a reversed rank: larger values mean greater use. Work and doing fractions are equal averages of the three Q2 monthly shares. They are not pooled message counts. Microsoft uses the March wave, and ILO uses the 2025 productivity level.

In [5]:
rank = pd.read_csv(paths["OpenAI rank"], keep_default_na=False)
rank["iso3"] = rank["country"].apply(iso2_to_iso3)
quarter_text = rank["quarter"].astype(str).str[:10]
q2_rank = quarter_text.eq("2026-04-01")
oa = rank.loc[q2_rank, ["iso3", "rank"]].dropna().rename(columns={"rank":"openai_rank"})
assert not oa["iso3"].duplicated().any()
oa["openai_use_score"] = -pd.to_numeric(oa["openai_rank"])

q2 = ["2026-04-01", "2026-05-01", "2026-06-01"]
work = pd.read_csv(paths["OpenAI work"], keep_default_na=False)
doing = pd.read_csv(paths["OpenAI doing"], keep_default_na=False)
w = work[work["month"].astype(str).str[:10].isin(q2) & work["work_related"].eq(1)]
a = doing[doing["month"].astype(str).str[:10].isin(q2) & doing["work_related"].eq(1)
          & doing["ask_do_express"].eq("doing")]
def monthly_share(frame, output_name):
    assert not frame.duplicated(["country", "month"]).any()
    # Some countries outside the selected cohort have suppressed monthly data.
    selected = frame[frame["country"].apply(iso2_to_iso3).isin(survey["iso3"])]
    assert selected.groupby("country")["month"].nunique().eq(3).all()
    value = frame.groupby("country", as_index=False)["share_of_messages"].mean()
    value["iso3"] = value["country"].apply(iso2_to_iso3)
    return value.rename(columns={"share_of_messages":output_name})[["iso3", output_name]].dropna()
w = monthly_share(w, "openai_work_share")
a = monthly_share(a, "openai_work_doing_share")

ms = pd.read_csv(paths["Microsoft"])
value_col = "Estimated share of working-age adults who use generative AI"
ms = ms[ms["Day"].astype(str).str.startswith("2026-03") & ms["Code"].notna()
        & ~ms["Code"].astype(str).str.startswith("OWID")]
ms = ms[["Code", value_col]].rename(columns={"Code":"iso3", value_col:"ms_adoption_pct"})
assert ms["iso3"].is_unique

ilo = pd.read_csv(paths["ILOSTAT"])
ilo["iso3"] = ilo["ref_area.label"].apply(name_to_iso3)
ilo["time"] = pd.to_numeric(ilo["time"])
prod = ilo[ilo["iso3"].notna() & ilo["time"].eq(2025)]
assert prod["iso3"].is_unique
prod = prod[["iso3", "obs_value", "time"]].rename(columns={"obs_value":"output_per_hour", "time":"output_year"})
print("OpenAI quarter values:", sorted(rank.loc[q2_rank, "quarter"].unique()))
print("Microsoft selected rows:",len(ms), "| ILO 2025 countries:",len(prod))

OpenAI quarter values: ['2026-04-01']
Microsoft selected rows: 147 | ILO 2025 countries: 178


## 5. Build the final comparison tables

The selected survey cohort provides the 31 rows. The representative flag identifies 17 of them. A one-to-one merge check stops repeated country keys from multiplying observations. The fixed map retains 121 countries. The matching counts describe this study, not each provider's full coverage.

In [6]:
analysis = survey.merge(claude, on="iso3", how="left", validate="one_to_one")
for table in [oa, ms, prod]:
    analysis = analysis.merge(table, on="iso3", how="left", validate="one_to_one")
deep = analysis.copy()
for table in [depth, w, a]:
    deep = deep.merge(table, on="iso3", how="left", validate="one_to_one")
assert len(deep) == 31 and deep["iso3"].is_unique
assert deep["ipsos_representative"].astype(str).str.lower().eq("true").sum() == 17
assert not deep.isna().any().any(), deep.isna().sum()
assert deep["output_year"].eq(2025).all()
map_table = claude.rename(columns={"claude_usage_share_pct":"usage_pct",
                                   "claude_per_capita_index":"usage_per_capita_index"})
analysis.to_csv(CLEAN / "analysis_table.csv", index=False)
deep.to_csv(CLEAN / "analysis_depth.csv", index=False)
map_table.to_csv(CLEAN / "usage_by_country.csv", index=False)
print("Saved analysis_table.csv, analysis_depth.csv and usage_by_country.csv.")
print("Rows:",len(deep), "| Representative countries:",17, "| Map countries:",len(map_table))
print("Missing values:",int(deep.isna().sum().sum()))

Saved analysis_table.csv, analysis_depth.csv and usage_by_country.csv.
Rows: 31 | Representative countries: 17 | Map countries: 121
Missing values: 0


## Next step

Run 02_analysis_and_exports.ipynb to calculate correlations, country means and group medians from these saved tables. The publication version keeps full available precision in its outputs. Tableau's older lookup may show minor rounding differences until refreshed; regenerating these files does not change the live workbook automatically.